# Inspect an instruction dataset

An `llm_finetune` dataset is records, not rows of text: `instruction` / `input` /
`output`, or a `messages` list for chat. What goes wrong is length — the answer
that gets cut off at `max_seq_length` is still counted as a training example,
and the model learns to stop mid-sentence.

In [ ]:
import orinth
import polars as pl

source = max(
    (dataset for dataset in orinth.datasets.list() if dataset.task_type == "llm_finetune"),
    key=lambda dataset: dataset.total_items,
)
print(source.id, "|", source.format, "|", source.splits)

frame = orinth.datasets.load(source.id, "train", limit=2000)
print(frame.columns)
frame.head(3)

## One record, in full

Read one before measuring any. The single most common surprise in an
instruction set is that the field you assumed held the answer holds the prompt.

In [ ]:
record = frame.row(0, named=True)
for key, value in record.items():
    rendered = str(value)
    print(f"--- {key} ---\n{rendered[:600]}{'…' if len(rendered) > 600 else ''}\n")

## Lengths

Prompt and response separately, because they truncate differently: a long prompt
costs context, a long response costs the part of the answer the model never sees
during training.

In [ ]:
if "messages" in frame.columns:
    rows = []
    for record in frame.iter_rows(named=True):
        messages = record["messages"] or []
        prompt = " ".join(str(m.get("content", "")) for m in messages if m.get("role") != "assistant")
        answer = " ".join(str(m.get("content", "")) for m in messages if m.get("role") == "assistant")
        rows.append({"turns": len(messages), "prompt": len(prompt), "answer": len(answer)})
    lengths = pl.DataFrame(rows)
else:
    lengths = frame.select(
        (pl.col("instruction").fill_null("") + " " + pl.col("input").fill_null("")).str.len_chars().alias("prompt"),
        pl.col("output").fill_null("").str.len_chars().alias("answer"),
    )

print(lengths.describe())

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(9, 3))
for axis, column in zip(axes, ["prompt", "answer"]):
    axis.hist(lengths[column].to_list(), bins=40)
    axis.set_xlabel(f"{column} characters")
figure.tight_layout()
plt.show()

# A rough budget: ~4 characters per token for English. Count with the real
# tokenizer once you have picked a base model — the ratio is nothing like
# constant across languages.
BUDGET_TOKENS = 2048
over = ((lengths["prompt"] + lengths["answer"]) / 4 > BUDGET_TOKENS).sum()
print(f"{over}/{lengths.height} records over a {BUDGET_TOKENS}-token budget")

## Roles, for a chat dataset

Every conversation should end with the assistant — that final turn is the only
thing being learned. One that ends on the user is a record with no target.

In [ ]:
if "messages" in frame.columns:
    endings = pl.DataFrame(
        [
            {"last_role": (record["messages"] or [{}])[-1].get("role", "missing")}
            for record in frame.iter_rows(named=True)
        ]
    )
    print(endings.group_by("last_role").agg(pl.len().alias("records")).sort("records", descending=True))
else:
    empty = frame.filter(pl.col("output").fill_null("").str.strip_chars() == "").height
    print(f"{empty}/{frame.height} records have an empty output — nothing to learn from those")